In [84]:
import csv
import json
import pandas as pd
import numpy as np

###Task 1.1 Read and clean the appointment data


In [85]:
# convert the required fields into int
def parse_data(row):
  row['age']=int(row['age'])
  row['fee_inr']=int(row['fee_inr'])
  row['prior_no_shows']=int(row['prior_no_shows'])
  return row
accepted=[]
rejected=[]
total_records=0
# parse the data and only accept cleaned records and reject the rest
with open("appointments.csv","r",newline="") as file :
  appointment=csv.DictReader(file)
  for row in appointment:
    total_records+=1
    try:
      converetd_appointments=parse_data(row)
      accepted.append(converetd_appointments)
    except ValueError:
      rejected.append(row)
# display the stats
print("appointments Header fields :",appointment.fieldnames)
print("Total records in appointments:",total_records)
print("Accepted rows:",len(accepted))
print("Rejected rows:",len(rejected))
with open("rejected.json","w") as file:
  json.dump(rejected,file,indent=2)

appointments Header fields : ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total records in appointments: 1020
Accepted rows: 1008
Rejected rows: 12


###Task 1.2 — Inspect and clean, pass 1

In [86]:
# Read and describe appointments
appointments=pd.read_csv("appointments.csv")
print("shape of appointments:",appointments.shape)
print("info of appointments:",appointments.info)
print("Missing values:",appointments.isnull().sum())
print("duplicate rows:",appointments.duplicated().sum())

# drop duplicates
appointments=appointments.drop_duplicates()
print("count After deleting duplicates:",len(appointments))

# clean booking_channel with .str.strip().str.title()
appointments['booking_channel']=appointments['booking_channel'].str.strip().str.title()
print("count After cleaning booking channel:",len(appointments))
appointments['appt_date']=pd.to_datetime(appointments['appt_date'],errors="coerce")
print("count After cleaning appt_date :",len(appointments))
appointments['booked_on']=pd.to_datetime(appointments['booked_on'],errors="coerce")
print("count After cleaning booked_on :",len(appointments))

# drop rows where appt_date is missing after conversion or status is missing
appointments=appointments.dropna(subset=['appt_date','status'])
print("drop rows where appt_date or status is missing :",len(appointments))






shape of appointments: (1020, 14)
info of appointments: <bound method DataFrame.info of      appointment_id patient_id clinic_id   booked_on   appt_date   age gender  \
0             A0001       P509       C05  2025-01-24  2025-01-25  60.0      F   
1             A0002       P046       C07  2025-05-27  2025-06-06  39.0      M   
2             A0003       P009       C02  2025-03-24  2025-03-30  63.0      F   
3             A0003       P009       C02  2025-03-24  2025-03-30  63.0      F   
4             A0004       P482       C01  2025-04-18  2025-04-22  52.0      F   
...             ...        ...       ...         ...         ...   ...    ...   
1015          A0996       P415       C05  2025-01-09  2025-01-31  20.0      M   
1016          A0997       P247       C05  2025-02-23  2025-02-27  77.0      M   
1017          A0998       P422       C06  2025-05-05  2025-05-16  58.0      M   
1018          A0999       P509       C05  2025-03-30  2025-04-12  59.0      F   
1019          A1000  

Task 1.3 — Clean, pass 2: validity, gaps and outliers

In [87]:
import warnings
warnings.filterwarnings('ignore')

In [88]:
# (a) Treat any age below 1 or above 100 as invalid and set it to missing
appointments['age'] = appointments['age'].where(
    appointments['age'].between(1, 100),
    pd.NA
)

# (b)Fill missing age with the median (then .round().astype(int)) and missing distance_km with its median.
appointments['age'] = appointments['age'].fillna(appointments['age'].median()).round().astype('Int64')
appointments['distance_km'] = appointments['distance_km'].fillna(appointments['distance_km'].median())

# (c) For distance_km and fee_inr, compute Q1, Q3 and IQR with .quantile, and cap values at Q3 + 1.5×IQR and Q1 − 1.5×IQR with np.clip.
q1=appointments['distance_km'].quantile(0.25)
q3=appointments['distance_km'].quantile(0.75)
iqr=q3-q1
appointments['distance_km']=np.clip(appointments['distance_km'],q1-1.5*iqr,q3+1.5*iqr)
q1=appointments['fee_inr'].quantile(0.25)
q3=appointments['fee_inr'].quantile(0.75)
iqr=q3-q1
appointments['fee_inr']=np.clip(appointments['fee_inr'],q1-1.5*iqr,q3+1.5*iqr)

# (d) Round distance_km to 1 decimal and fee_inr to whole rupees (.round().astype(int)).
appointments['distance_km'] = appointments['distance_km'].round(1)
appointments['fee_inr'] = appointments['fee_inr'].round().astype(int)

# Verify
print(appointments[['age', 'distance_km', 'fee_inr']].head())
print(appointments[['age', 'distance_km', 'fee_inr']].isnull().sum())



   age  distance_km  fee_inr
0   60         18.0      850
1   39         12.5      350
2   63          4.1      850
4   52          7.5      750
5   64         19.0      700
age            0
distance_km    0
fee_inr        0
dtype: int64


###Task 1.4 — Features and a NumPy summary

In [89]:
import numpy as np

# Create lead_days = days between booked_on and appt_date
appointments['lead_days'] = (
    appointments['appt_date'] - appointments['booked_on']
).dt.days


# Create no_show = 1 if status is "No-Show", otherwise 0
appointments['no_show'] = np.where(
    appointments['status'] == 'No-Show', 1, 0
)


# Select required columns
X = appointments[
    ["age", "distance_km", "fee_inr", "lead_days"]
]


# (a) Calculate the mean of each column (axis=0)
X_mean = np.mean(X, axis=0)

print("Mean of each column:\n", X_mean)


# (b) Calculate the number and share of rows with lead_days > 14
mask = X['lead_days'] > 14

print("Number of rows with lead_days > 14:", mask.sum())
print("Share of rows with lead_days > 14:", mask.mean())


# (c) Calculate z-scores of the distance_km column
z_score = (
    X['distance_km'] - np.mean(X['distance_km'])
) / np.std(X['distance_km'])

print("Z-scores of the distance_km column:\n", z_score.round(2))

# Count rows where absolute z-score > 2
print("Count of rows with |z| > 2:", (np.abs(z_score) > 2).sum())

Mean of each column:
 age             48.173737
distance_km      9.500909
fee_inr        635.714141
lead_days       11.819192
dtype: Float64
Number of rows with lead_days > 14: 296
Share of rows with lead_days > 14: 0.298989898989899
Z-scores of the distance_km column:
 0       1.38
1       0.49
2      -0.88
4      -0.32
5       1.54
        ... 
1015    0.57
1016    1.28
1017   -1.02
1018    2.61
1019    0.21
Name: distance_km, Length: 990, dtype: float64
Count of rows with |z| > 2: 53


###Task 1.5 — Enrich the clinics from a live API

In [90]:
import pandas as pd
import requests
import time

# Open-Meteo API URL
url = "https://api.open-meteo.com/v1/forecast"


# Function to fetch temperature for one clinic
def get_temperature(latitude, longitude):

    for attempt in range(2):  # First attempt + one retry

        try:
            response = requests.get(
                url,
                params={
                    "latitude": latitude,
                    "longitude": longitude,
                    "hourly": "temperature_2m",
                    "forecast_days": 1
                },
                timeout=10
            )

            # Check API response status
            if response.status_code != 200:
                reason = f"HTTP status code: {response.status_code}"
                continue

            # Read JSON response
            data = response.json()

            # Check whether hourly temperature data exists
            hourly = data.get("hourly")

            if not hourly or not hourly.get("temperature_2m"):
                reason = "Hourly temperature data is missing"
                continue

            # Take first temperature value (midnight UTC)
            temp_c = hourly["temperature_2m"][0]

            return temp_c

        except requests.exceptions.RequestException as e:
            reason = str(e)

        except ValueError as e:
            reason = f"Invalid JSON response: {e}"

    # Both attempts failed
    return None, reason

clinics=pd.read_csv("clinics.csv")
# Create a copy of the original DataFrame
clinics_enriched = clinics.copy()

# Store temperatures
temperatures = []

# Fetch temperature for each clinic
for _, row in clinics_enriched.iterrows():

    result = get_temperature(
        row['latitude'],
        row['longitude']
    )

    # Handle successful and failed API responses
    if isinstance(result, tuple):
        temp_c, reason = result

        print(f"Failed for {row['clinic_name']}: {reason}")

    else:
        temp_c = result

    temperatures.append(temp_c)


# Add temperature column
clinics_enriched['temp_c'] = temperatures


# Save enriched DataFrame to CSV
clinics_enriched.to_csv("clinics_enriched.csv", index=False)

print("\nEnriched clinic data:")
print(clinics_enriched)

print("\nSaved successfully as clinics_enriched.csv")


Enriched clinic data:
  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260   
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882   
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150   
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873   
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085   
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937   
7       C08    Gomti Nagar Care Point  Lucknow  Tier-2   26.8467    80.9462   

   temp_c  
0    17.7  
1    18.8  
2    20.8  
3    19.8  
4    19.8  
5    25.5  
6    18.7  
7    18.6  

Saved successfully as clinics_enriched.csv


###Task 1.6 — Export the clean table and write the SQL load script (4 marks)

In [91]:
# Sort the cleaned data by appointment_id
appointments.sort_values(by='appointment_id', inplace=True)
# Reset the index after sorting
appointments.reset_index(drop=True, inplace=True)

# # format both date columns as YYYY-MM-DD text (.dt.strftime("%Y-%m-%d"))
appointments['appt_date'] = appointments['appt_date'].dt.strftime("%Y-%m-%d")
appointments['booked_on'] = appointments['booked_on'].dt.strftime("%Y-%m-%d")
# # save appointments_clean.csv with exactly these 16 columns, in this order
# Define the exact column order
columns = [
    'appointment_id',
    'patient_id',
    'clinic_id',
    'booked_on',
    'appt_date',
    'age',
    'gender',
    'distance_km',
    'fee_inr',
    'booking_channel',
    'visit_type',
    'reminder_sent',
    'prior_no_shows',
    'status',
    'lead_days',
    'no_show'
]
appointments_clean = appointments.sort_values(
    by='appointment_id'
)[columns].copy()
appointments_clean.to_csv("appointments_clean.csv", index=False)
#loading the data as SQl
with open("load_data.sql", "w") as file:
  for _,row in clinics_enriched.iterrows():
    values = []
    for value in row:
      if pd.isna(value):
        values.append("NULL")
      elif isinstance(value,str):
        values.append(f"'{value}'")
      else:
        values.append(str(value))
    query2 = f"INSERT INTO clinics VALUES ({','.join(values)});\n"
    file.write(query2)
with open("load_data.sql", "a") as file:
  for _,row in appointments_clean.iterrows():
    values = []
    for value in row:
      if pd.isna(value):
        values.append("NULL")
      elif isinstance(value,str):
        values.append(f"'{value}'")
      else:
        values.append(str(value))
    query = f"INSERT INTO appointments VALUES ({','.join(values)});\n"
    file.write(query)


##Part 2 — SQL: verify the numbers and find where no-shows cluster

###Task 2.1 — Build and load the database

In [92]:
!pip install pymysql


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: C:\Users\chekuru keerthi\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [93]:
!pip install sqlalchemy


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: C:\Users\chekuru keerthi\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [94]:
import pymysql

connection = pymysql.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password="root",
    database="noshowguard"
    )
cursor = connection.cursor()

cursor.execute("SHOW DATABASES")

for db in cursor.fetchall():
    print(db)
print("MySQL connected successfully!")

('information_schema',)
('mysql',)
('noshowguard',)
('performance_schema',)
('sys',)
MySQL connected successfully!


###Task 2.2 — No-show rate by clinic

In [97]:
# Load the cleaned datasets
appointments = pd.read_csv("appointments_clean.csv")
clinics = pd.read_csv("clinics_enriched.csv")
# In Python, reproduce the clinic counts and no-show counts with groupby and confirm they match the SQL.

clinic_summary=(appointments.groupby("clinic_id").agg(appointments=("appointment_id","count"),no_shows=("no_show","sum")).reset_index())
# Add clinic name and city
clinic_summary=clinic_summary.merge(clinics,on="clinic_id",how="left")
# Calculate no-show percentage
clinic_summary["no_show_pct"]=(clinic_summary["no_shows"]/clinic_summary["appointments"]).round(1)
# Sort by no-show percentage
clinic_summary = clinic_summary.sort_values(
    "no_show_pct", ascending=False
)

display(clinic_summary)


,clinic_id,appointments,no_shows,clinic_name,city,tier,latitude,longitude,temp_c,no_show_pct
1,C02,143,39,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260,18.8,0.3
3,C04,142,38,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150,19.8,0.3
7,C08,90,30,Gomti Nagar Care Point,Lucknow,Tier-2,26.8467,80.9462,18.6,0.3
0,C01,160,36,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077,17.7,0.2
2,C03,82,13,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882,20.8,0.2
5,C06,142,23,Edappally Medical Centre,Kochi,Tier-2,10.0261,76.3085,25.5,0.2
6,C07,101,21,Vijay Nagar Clinic,Indore,Tier-2,22.7533,75.8937,18.7,0.2
4,C05,130,18,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873,19.8,0.1


In [98]:
# Keep only the clinics with at least 95 appointments
python_result = clinic_summary[
    clinic_summary["appointments"] >= 95
].copy()

print("Total appointments:", appointments.shape[0])
print("Total clinics:", clinics["clinic_id"].nunique())
print("Total no-shows:", appointments["no_show"].sum())

display(
    python_result[
        ["clinic_name", "city", "appointments",
         "no_shows", "no_show_pct"]
    ]
)
# Python results should match the SQL -->yes

Total appointments: 990
Total clinics: 8
Total no-shows: 218


,clinic_name,city,appointments,no_shows,no_show_pct
1,Hadapsar Care Centre,Pune,143,39,0.3
3,Malviya Nagar Clinic,Jaipur,142,38,0.3
0,Kothrud Family Clinic,Pune,160,36,0.2
5,Edappally Medical Centre,Kochi,142,23,0.2
6,Vijay Nagar Clinic,Indore,101,21,0.2
4,Vaishali Nagar Clinic,Jaipur,130,18,0.1
